# 02 Categories and Scope

This notebook asks a basic but powerful question:

> What changes when we count `DESAPARECIDA`, `NO LOCALIZADA`, and `CONFIDENCIAL` separately?

This matters because the legal categories are not just labels. They shape what the data can say.

## Legal Definitions

Under the Ley General en Materia de Desaparicion Forzada de Personas, Article 4:

- `Persona Desaparecida`: whereabouts unknown, with some indication that the absence is related to a crime.
- `Persona No Localizada`: location unknown, but the reported information does not indicate probable connection to a crime.

Our project's human-centered umbrella definition is broader:

> A person whose people cannot find them and do not know where they are.

We use official categories for measurement, but we do not let them fully define the human reality.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CSV = ROOT / "data" / "external" / "data.csv"
df = pd.read_csv(CSV, dtype="string")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
df.head()

## 1. What Status Categories Exist?

This is our first denominator decision. A national headline can include all statuses, but a violence-focused analysis may need to separate `DESAPARECIDA` from `NO LOCALIZADA`. `CONFIDENCIAL` is not a third human condition; it is a redaction/opacity condition.

In [ ]:
status_counts = (
    df["ESTATUS_VICTIMA"]
    .value_counts(dropna=False)
    .rename_axis("estatus")
    .reset_index(name="rows")
)
status_counts["share"] = (status_counts["rows"] / len(df)).round(4)
status_counts

## 2. What Data Is Available By Status?

A category can exist in the data but still be hard to analyze if key fields are missing. Here we check basic field availability.

In [ ]:
fields = [
    "FECHA_DESAPARICION",
    "FECHA_REGISTRO",
    "SEXO",
    "FECHA_NACIMIENTO",
    "ENTIDAD",
    "MUNICIPIO",
    "ORIGEN_REPORTE",
]

availability = []
for status, sub in df.groupby("ESTATUS_VICTIMA", dropna=False):
    row = {"estatus": status, "rows": len(sub)}
    for field in fields:
        missing = sub[field].isna() | (sub[field] == "CONFIDENCIAL") | (sub[field] == "")
        row[f"{field}_available"] = round((~missing).mean(), 3)
    availability.append(row)

pd.DataFrame(availability).sort_values("rows", ascending=False)

## 3. Category Mix By State

This is where opacity becomes geography. If one state has many `CONFIDENCIAL` rows, we may know the state total but lose sex, date, municipality, and status detail.

In [ ]:
state_status = pd.crosstab(df["ENTIDAD"], df["ESTATUS_VICTIMA"])
state_status["TOTAL"] = state_status.sum(axis=1)
state_status["CONFIDENCIAL_share"] = (state_status.get("CONFIDENCIAL", 0) / state_status["TOTAL"]).round(3)

state_status.sort_values("TOTAL", ascending=False).head(15)

## 4. What Would Be Fair To Claim?

Use this table to decide which denominator fits which claim. This prevents us from treating one dataset as if it answered every question.

In [ ]:
claim_denominators = pd.DataFrame([
    {
        "claim_type": "National unresolved stock",
        "recommended_denominator": "All rows, including CONFIDENCIAL",
        "reason": "Even redacted records represent unresolved people in this snapshot.",
    },
    {
        "claim_type": "Violence/crime-linked disappearance",
        "recommended_denominator": "DESAPARECIDA only, plus caveat about CONFIDENCIAL",
        "reason": "The legal category signals possible relation to crime; confidential rows may hide additional cases.",
    },
    {
        "claim_type": "Unknown-location / broader family-search burden",
        "recommended_denominator": "DESAPARECIDA + NO LOCALIZADA, with categories separated",
        "reason": "Both categories involve families/people not knowing where someone is, but they mean different things legally.",
    },
    {
        "claim_type": "Demographics by sex/age",
        "recommended_denominator": "Non-confidential rows with valid sex/age fields",
        "reason": "CONFIDENCIAL rows hide these details and cannot be distributed safely across categories.",
    },
    {
        "claim_type": "Trends by year",
        "recommended_denominator": "Rows with valid FECHA_DESAPARICION",
        "reason": "Year is unavailable for confidential rows; trends are stock, not all incident reports.",
    },
])

pd.set_option("display.max_colwidth", 120)
claim_denominators

## Exit Ticket

Answer these before moving on:

1. When should we include `NO LOCALIZADA`?
2. When should we exclude it or separate it?
3. What does `CONFIDENCIAL` prevent us from analyzing?
4. Which claim would be misleading if we used all rows as the denominator?
5. Which claim would be misleading if we ignored confidential rows completely?